In [1]:
from pathlib import Path

import duckdb


# The notebook is inside project_root/notebooks,
# so move one folder upward.
PROJECT_ROOT = Path.cwd().parent

# Define where the persistent DuckDB file should live.
DATABASE_PATH = (
    PROJECT_ROOT
    / "data"
    / "database"
    / "analytics.duckdb"
)

# Create data/database if it does not already exist.
DATABASE_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

# Open the existing database or create a new one.
connection = duckdb.connect(str(DATABASE_PATH))

print("Project root:", PROJECT_ROOT)
print("Database path:", DATABASE_PATH)
print("Database exists:", DATABASE_PATH.exists())

Project root: c:\Users\ozgur\Documents\GitHub\market-intelligence-pipeline
Database path: c:\Users\ozgur\Documents\GitHub\market-intelligence-pipeline\data\database\analytics.duckdb
Database exists: True


In [2]:
import pandas as pd

OLIST_DATA_DIRECTORY = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "olist"
)

olist_files = sorted(
    path.name
    for path in OLIST_DATA_DIRECTORY.glob("*.csv")
)

olist_files

['olist_customers_dataset.csv',
 'olist_geolocation_dataset.csv',
 'olist_order_items_dataset.csv',
 'olist_order_payments_dataset.csv',
 'olist_order_reviews_dataset.csv',
 'olist_orders_dataset.csv',
 'olist_products_dataset.csv',
 'olist_sellers_dataset.csv',
 'product_category_name_translation.csv']

In [3]:
ORDERS_CSV_PATH = (
    OLIST_DATA_DIRECTORY
    / "olist_orders_dataset.csv"
)

ORDER_ITEMS_CSV_PATH = (
    OLIST_DATA_DIRECTORY
    / "olist_order_items_dataset.csv"
)

print("Orders CSV:", ORDERS_CSV_PATH)
print("Order items CSV:", ORDER_ITEMS_CSV_PATH)

assert ORDERS_CSV_PATH.exists(), (
    f"Orders file not found: {ORDERS_CSV_PATH}"
)

assert ORDER_ITEMS_CSV_PATH.exists(), (
    f"Order-items file not found: {ORDER_ITEMS_CSV_PATH}"
)

Orders CSV: c:\Users\ozgur\Documents\GitHub\market-intelligence-pipeline\data\raw\olist\olist_orders_dataset.csv
Order items CSV: c:\Users\ozgur\Documents\GitHub\market-intelligence-pipeline\data\raw\olist\olist_order_items_dataset.csv


In [5]:
order_date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

orders = pd.read_csv(
    ORDERS_CSV_PATH,
    parse_dates=order_date_columns,
)

orders.head()


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


In [6]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 6.1 MB


In [7]:
order_items = pd.read_csv(
    ORDER_ITEMS_CSV_PATH,
    parse_dates=["shipping_limit_date"],
)

order_items.head()


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [8]:
order_items.info()

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   order_id             112650 non-null  str           
 1   order_item_id        112650 non-null  int64         
 2   product_id           112650 non-null  str           
 3   seller_id            112650 non-null  str           
 4   shipping_limit_date  112650 non-null  datetime64[us]
 5   price                112650 non-null  float64       
 6   freight_value        112650 non-null  float64       
dtypes: datetime64[us](1), float64(2), int64(1), str(3)
memory usage: 6.0 MB


In [9]:
print(
    "orders.order_id unique:",
    orders["order_id"].is_unique,
)

print(
    "order_items.order_id unique:",
    order_items["order_id"].is_unique,
)

print(
    "order_items composite key unique:",
    ~order_items.duplicated(
        subset=[
            "order_id",
            "order_item_id",
        ]
    ).any(),
)

orders.order_id unique: True
order_items.order_id unique: False
order_items composite key unique: True


In [10]:
assert orders["order_id"].is_unique

assert not order_items["order_id"].is_unique

assert not order_items.duplicated(
    subset=[
        "order_id",
        "order_item_id",
    ]
).any()

In [11]:
connection = duckdb.connect(str(DATABASE_PATH))

connection.execute("""
    SHOW TABLES
""").df()

,name


In [12]:
connection.register(
    "orders_df",
    orders,
)

connection.register(
    "order_items_df",
    order_items,
)

In [14]:
connection.execute("""
    SELECT
        order_status,
        COUNT(*) AS order_count
    FROM orders_df
    GROUP BY order_status
    ORDER BY order_count DESC
""").df()

,order_status,order_count
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


In [15]:
connection.execute("""
    CREATE OR REPLACE TABLE orders AS

    SELECT
        *
    FROM orders_df
""")

connection.execute("""
    CREATE OR REPLACE TABLE order_items AS

    SELECT
        *
    FROM order_items_df
""")

In [16]:
connection.execute("""
    SHOW TABLES
""").df()

,name
0,order_items
1,order_items_df
2,orders
3,orders_df


In [17]:
database_order_count = connection.execute("""
    SELECT
        COUNT(*) AS row_count
    FROM orders
""").fetchone()[0]

database_order_item_count = connection.execute("""
    SELECT
        COUNT(*) AS row_count
    FROM order_items
""").fetchone()[0]

print(
    "Pandas orders:",
    len(orders),
)

print(
    "DuckDB orders:",
    database_order_count,
)

print(
    "Pandas order items:",
    len(order_items),
)

print(
    "DuckDB order items:",
    database_order_item_count,
)

assert database_order_count == len(orders)

assert database_order_item_count == len(order_items)

Pandas orders: 99441
DuckDB orders: 99441
Pandas order items: 112650
DuckDB order items: 112650


In [18]:
connection.unregister("orders_df")
connection.unregister("order_items_df")

In [19]:
connection.execute("""
    SELECT
        COUNT(*) AS order_count
    FROM orders
""").df()

,order_count
0,99441


In [21]:
connection.execute("""
    CREATE OR REPLACE TABLE order_value AS

    SELECT
        order_id,

        SUM(
            price + freight_value
        ) AS gross_order_value,

        COUNT(
            order_item_id
        ) AS item_count,

        AVG(
            price + freight_value
        ) AS average_item_value

    FROM order_items

    GROUP BY order_id
""")

In [22]:
connection.execute("""
    SELECT
        *
    FROM order_value
    LIMIT 10
""").df()

,order_id,gross_order_value,item_count,average_item_value
0,000229ec398224ef6ca0657da4fc703e,216.87,1,216.87
1,0006ec9db01a64e59a68b2c340bf65a7,97.32,1,97.32
2,0008288aa423d2a3f00fcb17cd7d8719,126.54,2,63.27
3,00229e4e43f7a7e0b9dd819ad43268d3,91.39,1,91.39
4,00664284de7a3470d22931ed78615ee4,55.75,1,55.75
5,008c3a655c66f4d92370d0d422732f69,99.44,1,99.44
6,00c842d0ae6462afc1d38fc055b02c75,77.57,1,77.57
7,00d4439957b7fd3f8da4a33a965817fb,33.97,1,33.97
8,00eef7fe9e315a85d1a61a50cccb2494,63.27,1,63.27
9,00fcf938cde49ae138942b632ed62393,50.34,1,50.34


In [26]:
connection.execute("""
    CREATE OR REPLACE TABLE orders_enriched AS

    SELECT
        o.order_id,
        o.customer_id,
        o.order_status,
        o.order_purchase_timestamp,
        o.order_approved_at,
        o.order_delivered_carrier_date,
        o.order_delivered_customer_date,
        o.order_estimated_delivery_date,

        ov.gross_order_value,
        ov.item_count,
        ov.average_item_value

    FROM orders AS o

    LEFT JOIN order_value AS ov
        ON o.order_id = ov.order_id
""")

In [27]:
connection.execute("""
    CREATE OR REPLACE TABLE monthly_order_metrics AS

    SELECT
        CAST(
            DATE_TRUNC(
                'month',
                order_purchase_timestamp
            )
            AS DATE
        ) AS purchase_month,

        COUNT(
            DISTINCT order_id
        ) AS order_count,

        SUM(
            gross_order_value
        ) AS total_gross_order_value,

        AVG(
            gross_order_value
        ) AS average_gross_order_value,

        AVG(
            item_count
        ) AS average_item_count

    FROM orders_enriched

    WHERE order_status = 'delivered'

    GROUP BY purchase_month

    ORDER BY purchase_month
""")

In [28]:
monthly_metrics_df = connection.execute("""
    SELECT
        *
    FROM monthly_order_metrics
    ORDER BY purchase_month
""").df()

monthly_metrics_df

,purchase_month,order_count,total_gross_order_value,average_gross_order_value,average_item_count
0,2016-09-01,1,143.46,143.460000,3.000000
1,2016-10-01,265,46490.66,175.436453,1.181132
2,2016-12-01,1,19.62,19.620000,1.000000
3,2017-01-01,750,127482.37,169.976493,1.217333
4,2017-02-01,1653,271239.32,164.089123,1.124017
5,2017-03-01,2546,414330.95,162.738001,1.137863
6,2017-04-01,2303,390812.40,169.697091,1.115502
7,2017-05-01,3546,566851.40,159.856571,1.129160
8,2017-06-01,3135,490050.37,156.315907,1.112919
9,2017-07-01,3872,566299.08,146.254928,1.140496


In [29]:
total_orders = connection.execute("""
    SELECT
        COUNT(*) AS total_orders
    FROM orders
""").fetchone()

total_orders

(99441,)

In [30]:
status_counts = connection.execute("""
    SELECT
        order_status,
        COUNT(*) AS order_count
    FROM orders
    GROUP BY order_status
    ORDER BY order_count DESC
""").fetchall()

status_counts

[('delivered', 96478),
 ('shipped', 1107),
 ('canceled', 625),
 ('unavailable', 609),
 ('invoiced', 314),
 ('processing', 301),
 ('created', 5),
 ('approved', 2)]

In [31]:
status_query = """
    SELECT
        order_id,
        order_status,
        order_purchase_timestamp,
        gross_order_value,
        item_count

    FROM orders_enriched

    WHERE order_status = ?

    ORDER BY order_purchase_timestamp

    LIMIT ?
"""

delivered_sample = connection.execute(
    status_query,
    [
        "delivered",
        10,
    ],
).df()

delivered_sample

,order_id,order_status,order_purchase_timestamp,gross_order_value,item_count
0,bfbd0f9bdef84302105ad712db648a6c,delivered,2016-09-15 12:16:38,143.46,3
1,3b697a20d9e427646d92567910af6d57,delivered,2016-10-03 09:44:50,45.46,1
2,be5bc2f0da14d8071e2d45451ad119d9,delivered,2016-10-03 16:56:50,39.09,1
3,a41c8759fbe7aab36ea07e038b2d4465,delivered,2016-10-03 21:13:36,53.73,1
4,d207cc272675637bfed0062edffd0818,delivered,2016-10-03 22:06:03,133.46,1
5,cd3b8574c82b42fc8129f6d502690c3e,delivered,2016-10-03 22:31:31,40.95,1
6,ae8a60e4b03c5a4ba9ca0672c164b181,delivered,2016-10-03 22:44:10,154.57,1
7,ef1b29b591d31d57c0d7337460dd83c9,delivered,2016-10-03 22:51:30,92.27,1
8,0a0837a5eee9e7a9ce2b1fa831944d27,delivered,2016-10-04 09:06:10,65.50,1
9,1ff217aa612f6cd7c4255c9bfe931c8b,delivered,2016-10-04 09:16:33,44.23,1


In [32]:
connection.execute("""
    CREATE OR REPLACE VIEW delivered_orders AS

    SELECT
        *

    FROM orders_enriched

    WHERE order_status = 'delivered'
""")

In [33]:
connection.execute("""
    SELECT
        COUNT(*) AS delivered_order_count
    FROM delivered_orders
""").df()

,delivered_order_count
0,96478


In [34]:
connection.execute("""
    SHOW TABLES
""").df()

,name
0,delivered_orders
1,monthly_order_metrics
2,order_items
3,order_value
4,orders
5,orders_enriched


In [35]:
connection.execute("""
    SELECT
        table_name,
        table_type

    FROM information_schema.tables

    WHERE table_schema = 'main'

    ORDER BY table_name
""").df()

,table_name,table_type
0,delivered_orders,VIEW
1,monthly_order_metrics,BASE TABLE
2,order_items,BASE TABLE
3,order_value,BASE TABLE
4,orders,BASE TABLE
5,orders_enriched,BASE TABLE


In [38]:
connection.close()